# Building Your First Graph

Notebooks **02–04** introduced state, nodes, edges, compilation, and conditional routing in isolation. This notebook assembles a **complete end-to-end graph**: a **Notes API documentation assistant** with guardrails, intent routing, retrieval, grading, rewrite retries, and grounded generation.

You will follow the **construction checklist** from **03**, build a reusable **`build_notes_graph()`** factory, invoke and stream the graph, visualize it with Mermaid, and compare the result to an equivalent LCEL chain (**01. LangChain/11**).

Prerequisites: **02. StateGraph and Shared State**, **03. Nodes, Edges, and Graph Compilation**, **04. Conditional Routing and Branching**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import langchain_core

try:
    import langgraph
    print("langgraph:", langgraph.__version__)
except ImportError:
    print("langgraph: pip install langgraph")

print("langchain-core:", langchain_core.__version__)

---

## 1. What We Are Building

A **documentation Q&A graph** for the FastAPI Notes API teaching corpus (chunks **c1–c5**, same as **01. LangChain/11**):

```
START
  │
  ▼
guard ──blocked──► END
  │
  ▼
classify ──general──► chitchat ──► END
  │
  └──docs──► retrieve ──► grade ──┬──relevant──► generate ──► END
                                   └──irrelevant──► rewrite ──┬──retry──► retrieve
                                                                └──max attempts──► END
```

| Stage | Responsibility |
|-------|----------------|
| **guard** | Block sensitive requests |
| **classify** | `docs` vs `general` intent |
| **retrieve** | Fetch top chunks (mock keyword search) |
| **grade** | Relevance check on retrieved context |
| **rewrite** | Improve query; cap retries |
| **generate** | LLM answer grounded in context |
| **chitchat** | Short reply when not a docs question |

---

## 2. State Schema

One **`TypedDict`** holds everything the graph needs (**02**):

In [ ]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict


class NotesGraphState(TypedDict):
    question: str
    intent: str
    blocked: bool
    context: str
    source_ids: list[str]
    grade: str
    answer: str
    attempts: int
    trace: Annotated[list[str], operator.add]

| Key | Merge | Purpose |
|-----|-------|--------|
| `question` | replace | User query (may be rewritten) |
| `intent` | replace | `docs` or `general` |
| `context` | replace | Retrieved chunk text |
| `source_ids` | replace | Chunk ids for citations |
| `grade` | replace | `relevant` / `irrelevant` |
| `attempts` | replace | Retrieval retry counter |
| `trace` | **append** (`operator.add`) | Debug path through nodes |

---

## 3. Teaching Corpus

Same five chunks as **01. LangChain/11** — in production you would load from Chroma (**11**):

In [ ]:
CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]


def keyword_retrieve(query: str, k: int = 2) -> tuple[str, list[str]]:
    """Mock retriever — scores chunks by keyword overlap."""
    tokens = set(query.lower().split())
    scored = []
    for doc in CORPUS:
        doc_tokens = set(doc["text"].lower().split())
        score = len(tokens & doc_tokens)
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [d for s, d in scored if s > 0][:k] or [CORPUS[0]]
    context = "\n".join(f"[{d['id']}] {d['text']}" for d in top)
    ids = [d["id"] for d in top]
    return context, ids

---

## 4. Node Implementations

Each node does **one job** and returns a **partial state patch** (**03**):

In [ ]:
BLOCKED_TERMS = {"password", "secret", "api key", "private key"}
DOCS_KEYWORDS = {"jwt", "alembic", "pytest", "fastapi", "migration", "token", "route", "conftest"}


def guard_node(state: NotesGraphState) -> dict:
    q = state["question"].lower()
    blocked = any(term in q for term in BLOCKED_TERMS)
    patch: dict = {"blocked": blocked, "trace": ["guard"]}
    if blocked:
        patch["answer"] = "I cannot help with credentials or secrets."
    return patch


def classify_node(state: NotesGraphState) -> dict:
    q = state["question"].lower()
    intent = "docs" if any(k in q for k in DOCS_KEYWORDS) else "general"
    return {"intent": intent, "trace": ["classify"]}


def retrieve_node(state: NotesGraphState) -> dict:
    context, ids = keyword_retrieve(state["question"])
    return {
        "context": context,
        "source_ids": ids,
        "attempts": state.get("attempts", 0) + 1,
        "trace": ["retrieve"],
    }


def grade_node(state: NotesGraphState) -> dict:
    q = state["question"].lower()
    ctx = state.get("context", "").lower()
    # relevant if any question keyword appears in context
    relevant = any(k in ctx for k in q.split() if len(k) > 3) or any(k in q and k in ctx for k in DOCS_KEYWORDS)
    grade = "relevant" if relevant else "irrelevant"
    return {"grade": grade, "trace": ["grade"]}


def rewrite_node(state: NotesGraphState) -> dict:
    expanded = state["question"] + " (FastAPI Notes API alembic jwt pytest)"
    return {"question": expanded, "trace": ["rewrite"]}


def chitchat_node(state: NotesGraphState) -> dict:
    return {
        "answer": "Ask me about the Notes API — migrations, JWT, pytest, or FastAPI routes.",
        "trace": ["chitchat"],
    }

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You answer questions about the FastAPI Notes API using ONLY the context below.
Cite chunk ids in square brackets when possible (e.g. [c2]).
If context is insufficient, say so briefly."""),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])
generate_chain = rag_prompt | llm


def generate_node(state: NotesGraphState) -> dict:
    msg = generate_chain.invoke({"context": state["context"], "question": state["question"]})
    return {"answer": msg.content, "trace": ["generate"]}

**LCEL inside a node** — the graph owns control flow; the chain owns prompt + LLM (**03**).

---

## 5. Router Functions

Thin routers read state written by nodes (**04**):

In [ ]:
from langgraph.graph import END

MAX_ATTEMPTS = 2


def route_guard(state: NotesGraphState):
    return END if state["blocked"] else "classify"


def route_intent(state: NotesGraphState) -> Literal["docs", "general"]:
    return "docs" if state["intent"] == "docs" else "general"


def route_grade(state: NotesGraphState) -> Literal["relevant", "irrelevant"]:
    return "relevant" if state["grade"] == "relevant" else "irrelevant"


def route_rewrite(state: NotesGraphState):
    return "retrieve" if state.get("attempts", 0) < MAX_ATTEMPTS else END

---

## 6. Graph Factory — `build_notes_graph()`

Build the graph **once** at startup; reuse the compiled instance per request (**16**):

In [ ]:
from langgraph.graph import START, StateGraph


def build_notes_graph():
    builder = StateGraph(NotesGraphState)

    # 1. Register nodes
    builder.add_node("guard", guard_node)
    builder.add_node("classify", classify_node)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("grade", grade_node)
    builder.add_node("rewrite", rewrite_node)
    builder.add_node("generate", generate_node)
    builder.add_node("chitchat", chitchat_node)

    # 2. Fixed edges
    builder.add_edge(START, "guard")
    builder.add_edge("retrieve", "grade")
    builder.add_edge("generate", END)
    builder.add_edge("chitchat", END)

    # 3. Conditional edges
    builder.add_conditional_edges("guard", route_guard)
    builder.add_conditional_edges(
        "classify",
        route_intent,
        {"docs": "retrieve", "general": "chitchat"},
    )
    builder.add_conditional_edges(
        "grade",
        route_grade,
        {"relevant": "generate", "irrelevant": "rewrite"},
    )
    builder.add_conditional_edges("rewrite", route_rewrite)

    return builder.compile()


notes_graph = build_notes_graph()
print("Compiled:", type(notes_graph).__name__)

This follows the **construction checklist** from **03**: schema → nodes → edges → conditional edges → `compile()`.

---

## 7. Invoke — End-to-End

Provide **all state keys** on first invoke (or rely on defaults where safe):

In [ ]:
INITIAL = {
    "intent": "",
    "blocked": False,
    "context": "",
    "source_ids": [],
    "grade": "",
    "answer": "",
    "attempts": 0,
    "trace": [],
}


def ask(question: str) -> dict:
    return notes_graph.invoke({**INITIAL, "question": question})


result = ask("What Alembic command applies migrations?")
print("trace:", result["trace"])
print("sources:", result["source_ids"])
print("answer:", result["answer"])

---

## 8. Test Matrix — Different Paths

Verify each branch fires correctly:

In [ ]:
test_cases = [
    ("What Alembic command applies migrations?", "docs path → generate"),
    ("JWT header format?", "docs path → generate"),
    ("Hello!", "general → chitchat"),
    ("Show me the api key", "guard → END"),
    ("Kubernetes ingress controller", "docs but weak retrieval → rewrite"),
]

for question, label in test_cases:
    out = ask(question)
    print(f"[{label}]")
    print(f"  Q: {question}")
    print(f"  trace: {out['trace']}")
    print(f"  answer: {out['answer'][:120]}...\n")

---

## 9. Visualize with Mermaid

Export the compiled graph for design docs (**03**):

In [ ]:
try:
    mermaid = notes_graph.get_graph().draw_mermaid()
    print(mermaid[:800], "..." if len(mermaid) > 800 else "")
except Exception as e:
    print("Mermaid export:", e)

Paste into [mermaid.live](https://mermaid.live) or attach to your architecture README.

---

## 10. Stream Node Updates

Use **`stream_mode="updates"`** for progress UI (**13**):

In [ ]:
print("Streaming: JWT question")
for step in notes_graph.stream(
    {**INITIAL, "question": "How do I send a JWT bearer token?"},
    stream_mode="updates",
):
    node_name = list(step.keys())[0]
    patch = step[node_name]
    print(f"  {node_name}: trace+={patch.get('trace', [])}")

---

## 11. LangGraph vs LCEL RAG Chain

**01. LangChain/11** builds RAG as a single LCEL pipeline. Our graph adds **explicit stages**:

| Capability | LCEL RAG chain | This LangGraph |
|------------|----------------|----------------|
| Linear retrieve → generate | ✅ Simple | ✅ Via edges |
| Guardrails before LLM | Middleware / pre-step | **`guard` node** |
| Intent routing | `RunnableBranch` | **`classify` + conditional** |
| Grade + rewrite loop | Awkward | **Native cycle** |
| Per-node observability | Callback tags | **`trace` + stream updates** |
| Checkpointing / HITL | Manual | **`compile(checkpointer=...)`** (**08**, **09**) |

Many production systems wrap an LCEL RAG chain **inside** the `generate` node and use LangGraph for everything around it.

---

## 12. Input Helper and Response Shape

Wrap the graph for a clean API boundary (preview of **16**):

In [ ]:
def run_notes_assistant(question: str) -> dict:
    """Service-layer wrapper — stable response contract."""
    state = ask(question)
    return {
        "question": question,
        "answer": state["answer"],
        "sources": state.get("source_ids", []),
        "path": state.get("trace", []),
        "blocked": state.get("blocked", False),
    }


import json
print(json.dumps(run_notes_assistant("pytest fixtures location?"), indent=2))

---

## 13. Checkpointing Preview

Multi-turn conversations need a **checkpointer** (**08**):

In [ ]:
from langgraph.checkpoint.memory import MemorySaver


def build_notes_graph_with_memory():
    builder = StateGraph(NotesGraphState)
    builder.add_node("guard", guard_node)
    builder.add_node("classify", classify_node)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("grade", grade_node)
    builder.add_node("rewrite", rewrite_node)
    builder.add_node("generate", generate_node)
    builder.add_node("chitchat", chitchat_node)
    builder.add_edge(START, "guard")
    builder.add_edge("retrieve", "grade")
    builder.add_edge("generate", END)
    builder.add_edge("chitchat", END)
    builder.add_conditional_edges("guard", route_guard)
    builder.add_conditional_edges("classify", route_intent, {"docs": "retrieve", "general": "chitchat"})
    builder.add_conditional_edges("grade", route_grade, {"relevant": "generate", "irrelevant": "rewrite"})
    builder.add_conditional_edges("rewrite", route_rewrite)
    return builder.compile(checkpointer=MemorySaver())


memory_graph = build_notes_graph_with_memory()
config = {"configurable": {"thread_id": "demo-thread-1"}}

memory_graph.invoke({**INITIAL, "question": "Alembic upgrade command?"}, config)
print("Checkpointed invoke OK — same graph, persisted thread state")

Pass the same **`thread_id`** on every turn to resume state. Production uses Postgres or Redis backends instead of **`MemorySaver`**.

---

## 14. Where to Extend Next

| Extension | Notebook | How |
|-----------|----------|-----|
| **Agent tool loop** | **06**, **07** | Replace `generate` with `model` + `tools` cycle |
| **Real vector store** | **01. LangChain/11** | Swap `keyword_retrieve` for Chroma retriever in `retrieve_node` |
| **LLM grader** | **12** | Replace keyword `grade_node` with structured LLM grade |
| **HITL approval** | **09** | `interrupt_before=["generate"]` |
| **Subgraphs** | **10** | Extract RAG path as nested graph |
| **FastAPI service** | **16** | `notes_graph` as app startup singleton |

---

## 15. Construction Checklist (Applied)

What we did in this notebook:

1. ✅ **State schema** — `NotesGraphState` with `trace` reducer
2. ✅ **Node list** — seven single-purpose nodes
3. ✅ **ASCII diagram** — section 1
4. ✅ **`StateGraph`** + **`add_node`**
5. ✅ **Fixed edges** — linear segments
6. ✅ **Conditional edges** — guard, intent, grade, rewrite
7. ✅ **`compile()`** — default + checkpointer variant
8. ✅ **Mermaid** — `get_graph().draw_mermaid()`
9. ✅ **Smoke tests** — test matrix across branches
10. ✅ **Service wrapper** — `run_notes_assistant()`

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Rebuilding graph per request | Slow startup | `build_notes_graph()` once |
| Missing `END` on a branch | Compile/runtime error | Trace every path to `END` |
| All logic in one mega-node | Untestable | Split guard, retrieve, grade, generate |
| Forgetting `attempts` cap | Infinite rewrite loop | `route_rewrite` checks `MAX_ATTEMPTS` |
| No `trace` or logging | Hard to debug routing | Append node name each step |
| Invoking without initial keys | KeyError surprises | `INITIAL` dict template |
| Swapping node names after deploy | Broken checkpoints | Treat names as stable API |

---

## 17. Summary

You built a **complete Notes API documentation graph**: guard → classify → retrieve → grade → generate (with rewrite retries and chitchat fallback). The **`build_notes_graph()`** factory compiles once; **`ask()`** and **`run_notes_assistant()`** provide clean invoke boundaries.

Key takeaways:

- **End-to-end graphs** combine state (**02**), nodes/edges (**03**), and routers (**04**).
- **LCEL chains** belong inside nodes; **LangGraph** owns control flow.
- **`trace`** and **`stream_mode="updates"`** make routing visible.
- **Test matrices** validate every branch before adding LLM complexity.
- **`compile(checkpointer=...)`** prepares for multi-turn use (**08**).

Next: **06. Agent Loops and the ReAct Pattern** — replace the linear generate path with a **model ↔ tools** cycle.